In [ ]:
#Values for Pressure on the Center 
Pc_vals = np.geomspace(1e-5, 3e3, 260)

#Plot
fig = plt.figure(figsize=(7,6))

#Pure hadronic curve
R_had, M_had = [], []    #Void Lists 
for Pc in Pc_vals:
    Rf, Mf = integrate_star_hadronic(Pc, P_R=1e-12)
    #Save data
    R_had.append(Rf)
    M_had.append(Mf / Msun_in_km)
    
# Plot pure hadronic FIRST
plt.plot(R_had, M_had, color='black',lw=2, zorder=10, label="Pure hadronic (Crust+APR-1)")

#Store curves in column format
curves = []
#Hadronic
curves.append(R_had)
curves.append(M_had)

#Color mapping based on Ptr values
ptr_colmaps = {
    35: cm.Blues,
    40: cm.Reds,
    45: cm.Purples,
    50: cm.Greens}

for Ptr in [35, 40, 45, 50]:
    Etr = APR_1(Ptr)
    DEcrit = 0.5 * Etr + 1.5 * Ptr
    dE_vals = [DEcrit, DEcrit + 100, DEcrit + 200]

    colmap = ptr_colmaps[Ptr]
    colors = colmap(np.linspace(0.5, 0.9, len(dE_vals)))  #Color gradient: light to dark

    for dE, col in zip(dE_vals, colors):
        R_list, M_list = [], []
        for Pc in Pc_vals:
            R_hyb, M_hyb = integrate_star_hybrid(Pc, Ptr=Ptr, DE=dE, cs2=1.0, P_R=1e-12)
            R_list.append(R_hyb)
            M_list.append(M_hyb)

        R_arr = np.array(R_list)
        M_arr = np.array(M_list) / Msun_in_km

        curves.append(np.array(R_arr))
        curves.append(np.array(M_arr))

        plt.plot(R_arr, M_arr, color=col, lw=1.4,label=f"Ptr={Ptr:.0f}, ΔE={dE:.0f}")

plt.xlim(6, 18)
plt.grid(True)
plt.legend(fontsize=8)
plt.xlabel("Radius R (km)")
plt.ylabel("Mass M / M$_\\odot$")
plt.title("TOV - Hybrid Star: Quark and Hadronic (APR-1) Matter (cs²=1)")
fig.savefig('Hybrid-Hadronic Matter', dpi=80, bbox_inches='tight')
plt.show()

data_out = np.column_stack(curves)

np.savetxt(
    "MR_ALL_curves_multi_column.txt",
    data_out,
    fmt="%.8e",
    header="R_had M_had R1 M1 R2 M2 ...",
    comments="")